In [1]:
# ============================================================
# NOTEBOOK 10C
# CXR CoAtNet Final Training — Run 2
#
# CELL 1 — FINAL DEVELOPMENT / TRAINING SAFETY GATE
# ============================================================

import os
import json
import hashlib
import random
import numpy as np
import pandas as pd
import torch
import timm

from pathlib import Path

# ------------------------------------------------------------
# 1. REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 80)
print("NOTEBOOK 10C — CoAtNet FINAL TRAINING RUN 2")
print("=" * 80)

print(f"PyTorch version : {torch.__version__}")
print(f"timm version    : {timm.__version__}")
print(f"Device          : {'CUDA' if torch.cuda.is_available() else 'CPU'}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------------------------------------
# 2. AUTHORITATIVE PATHS
# ------------------------------------------------------------

BASE_ROOT = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation"
)

FINAL_COHORT_ROOT = (
    BASE_ROOT
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

INPUT_ROOT = (
    FINAL_COHORT_ROOT
    / "CoAtNet_224_Final_Input"
    / "NPY"
)

MANIFEST_PATH = (
    FINAL_COHORT_ROOT
    / "CoAtNet_224_Final_Input"
    / "QC"
    / "Notebook08_Cell26B_FINAL_1976_CoAtNet_Input_Manifest.csv"
)

SPLIT_ROOT = (
    FINAL_COHORT_ROOT
    / "MASTER_CONDITION_LEVEL_SPLIT"
)

TRAIN_SPLIT = (
    SPLIT_ROOT
    / "Notebook08_Cell25_TRAIN.csv"
)

VAL_SPLIT = (
    SPLIT_ROOT
    / "Notebook08_Cell25_VALIDATION.csv"
)

TEST_SPLIT = (
    SPLIT_ROOT
    / "Notebook08_Cell25_TEST.csv"
)

OUTPUT_ROOT = (
    FINAL_COHORT_ROOT
    / "CoAtNet_Training_Run2_10C"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("\nPATH CHECK")
print("-" * 80)

required_paths = {
    "Final cohort root": FINAL_COHORT_ROOT,
    "Input NPY root": INPUT_ROOT,
    "Input manifest": MANIFEST_PATH,
    "Train split": TRAIN_SPLIT,
    "Validation split": VAL_SPLIT,
    "Test split": TEST_SPLIT,
}

for name, path in required_paths.items():
    status = "PASS" if path.exists() else "FAIL"
    print(f"{name:<25}: {status}")
    print(f"  {path}")


# ------------------------------------------------------------
# 3. LOAD AUTHORITATIVE SPLITS
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_SPLIT)
val_df = pd.read_csv(VAL_SPLIT)
test_df = pd.read_csv(TEST_SPLIT)

TARGET_COL = "target_binary"
CONDITION_COL = "condition_id"

assert TARGET_COL in train_df.columns
assert TARGET_COL in val_df.columns
assert TARGET_COL in test_df.columns

assert CONDITION_COL in train_df.columns
assert CONDITION_COL in val_df.columns
assert CONDITION_COL in test_df.columns


# ------------------------------------------------------------
# 4. BASIC SPLIT VALIDATION
# ------------------------------------------------------------

train_conditions = set(train_df[CONDITION_COL].astype(str))
val_conditions = set(val_df[CONDITION_COL].astype(str))
test_conditions = set(test_df[CONDITION_COL].astype(str))

train_val_overlap = train_conditions & val_conditions
train_test_overlap = train_conditions & test_conditions
val_test_overlap = val_conditions & test_conditions

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

print("\nSPLIT COUNTS")
print("-" * 80)

print(f"Train      : {len(train_df)}")
print(f"Validation : {len(val_df)}")
print(f"Test       : {len(test_df)}")

print("\nTARGET DISTRIBUTION")
print("-" * 80)

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    counts = df[TARGET_COL].value_counts().sort_index()

    print(
        f"{name:<12} "
        f"DS={int(counts.get(0, 0)):<5} "
        f"DR={int(counts.get(1, 0)):<5}"
    )


# ------------------------------------------------------------
# 5. EXPECTED FROZEN COUNTS
# ------------------------------------------------------------

assert len(train_df) == 1185
assert len(val_df) == 395
assert len(test_df) == 396

assert train_df[TARGET_COL].sum() == 763
assert val_df[TARGET_COL].sum() == 254
assert test_df[TARGET_COL].sum() == 255


# ------------------------------------------------------------
# 6. PHYSICAL INPUT VERIFICATION
# ------------------------------------------------------------

physical_npy = {
    p.stem
    for p in INPUT_ROOT.glob("*.npy")
}

print("\nPHYSICAL INPUT")
print("-" * 80)
print(f"NPY files found : {len(physical_npy)}")

assert len(physical_npy) == 1976


# ------------------------------------------------------------
# 7. TRAIN + VALIDATION INPUT AVAILABILITY
# ------------------------------------------------------------

development_df = pd.concat(
    [train_df, val_df],
    ignore_index=True
)

missing_development = []

for condition_id in development_df[CONDITION_COL].astype(str):
    if condition_id not in physical_npy:
        missing_development.append(condition_id)

assert len(missing_development) == 0

print("Train/validation physical input check : PASS")


# ------------------------------------------------------------
# 8. TEST ISOLATION
# ------------------------------------------------------------

# Test IDs are checked only for overlap safety.
# NO TEST IMAGE WILL BE LOADED BY THE TRAINING DATASET.

print("\nTEST ISOLATION")
print("-" * 80)
print("Test data used for training      : NO")
print("Test data used for augmentation  : NO")
print("Test data used for model choice  : NO")
print("Test data used for thresholding  : NO")
print("Test predictions during training: NO")


# ------------------------------------------------------------
# 9. IMAGE NORMALIZATION POLICY
# ------------------------------------------------------------

# IMPORTANT:
# CoAtNet uses ImageNet-pretrained weights.
# Therefore the final tensor normalization will use
# ImageNet statistics.

IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)

print("\nNORMALIZATION POLICY")
print("-" * 80)
print("ImageNet mean:", IMAGENET_MEAN)
print("ImageNet std :", IMAGENET_STD)


# ------------------------------------------------------------
# 10. TARGET PERFORMANCE
# ------------------------------------------------------------

TARGET_TRAIN_ACCURACY = 0.90
TARGET_VAL_ACCURACY = 0.90
TARGET_VAL_ROC_AUC = 0.90

print("\nDEVELOPMENT TARGET")
print("-" * 80)
print(f"Train accuracy target      : > {TARGET_TRAIN_ACCURACY * 100:.0f}%")
print(f"Validation accuracy target : > {TARGET_VAL_ACCURACY * 100:.0f}%")
print(f"Validation ROC-AUC target  : > {TARGET_VAL_ROC_AUC * 100:.0f}%")


# ------------------------------------------------------------
# 11. SAVE CONFIGURATION
# ------------------------------------------------------------

config = {
    "notebook": "10C_CXR_CoAtNet_Final_Training_Run2",
    "seed": SEED,
    "device": str(DEVICE),

    "train_records": len(train_df),
    "validation_records": len(val_df),
    "test_records": len(test_df),

    "train_DS": int((train_df[TARGET_COL] == 0).sum()),
    "train_DR": int((train_df[TARGET_COL] == 1).sum()),

    "validation_DS": int((val_df[TARGET_COL] == 0).sum()),
    "validation_DR": int((val_df[TARGET_COL] == 1).sum()),

    "test_DS": int((test_df[TARGET_COL] == 0).sum()),
    "test_DR": int((test_df[TARGET_COL] == 1).sum()),

    "target_column": TARGET_COL,

    "imagenet_mean": IMAGENET_MEAN.tolist(),
    "imagenet_std": IMAGENET_STD.tolist(),

    "test_used_for_training": False,
    "test_used_for_model_selection": False,
    "test_used_for_threshold_tuning": False,

    "target_train_accuracy": TARGET_TRAIN_ACCURACY,
    "target_validation_accuracy": TARGET_VAL_ACCURACY,
    "target_validation_roc_auc": TARGET_VAL_ROC_AUC,
}

config_path = (
    OUTPUT_ROOT
    / "Notebook10C_Cell1_Run2_Safety_Gate.json"
)

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("\n" + "=" * 80)
print("CELL 1 STATUS: PASS")
print("=" * 80)

NOTEBOOK 10C — CoAtNet FINAL TRAINING RUN 2
PyTorch version : 2.13.0+cpu
timm version    : 1.0.28
Device          : CPU

PATH CHECK
--------------------------------------------------------------------------------
Final cohort root        : PASS
  C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT
Input NPY root           : PASS
  C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\CoAtNet_224_Final_Input\NPY
Input manifest           : PASS
  C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\CoAtNet_224_Final_Input\QC\Notebook08_Cell26B_FINAL_1976_CoAtNet_Input_Manifest.csv
Train split              : PASS
  C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation\QC\Clean_PSPNet_CXR_Cohort\FINAL_FROZEN_CLEAN_COHORT\MASTER_CONDITION_LEVEL_SPLIT\Notebook08_Cell25_TRAIN.csv
Validation split         : PASS
  C:\TBP\Metadata\Step_3B_8_CXR_CoAtNe

In [2]:
# ============================================================
# NOTEBOOK 10C
# CXR CoAtNet FINAL TRAINING — RUN 2
#
# CELL 2 — DATASET + AUGMENTATION PIPELINE
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("NOTEBOOK 10C — CELL 2")
print("FINAL RUN-2 DATASET + AUGMENTATION PIPELINE")
print("=" * 80)

print(f"Device : {DEVICE}")


# ============================================================
# 3. PATHS
# ============================================================

BASE_ROOT = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation"
)

FINAL_COHORT_ROOT = (
    BASE_ROOT
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

INPUT_ROOT = (
    FINAL_COHORT_ROOT
    / "CoAtNet_224_Final_Input"
    / "NPY"
)

SPLIT_ROOT = (
    FINAL_COHORT_ROOT
    / "MASTER_CONDITION_LEVEL_SPLIT"
)

TRAIN_SPLIT = (
    SPLIT_ROOT
    / "Notebook08_Cell25_TRAIN.csv"
)

VAL_SPLIT = (
    SPLIT_ROOT
    / "Notebook08_Cell25_VALIDATION.csv"
)

TEST_SPLIT = (
    SPLIT_ROOT
    / "Notebook08_Cell25_TEST.csv"
)

OUTPUT_ROOT = (
    FINAL_COHORT_ROOT
    / "CoAtNet_Training_Run2_10C"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. LOAD SPLITS
# ============================================================

train_df = pd.read_csv(TRAIN_SPLIT)
val_df = pd.read_csv(VAL_SPLIT)

# Test dataframe is deliberately NOT loaded.
# Test remains isolated until final evaluation.

TARGET_COL = "target_binary"
CONDITION_COL = "condition_id"

assert TARGET_COL in train_df.columns
assert TARGET_COL in val_df.columns

assert CONDITION_COL in train_df.columns
assert CONDITION_COL in val_df.columns

print("\nSPLIT CHECK")
print("-" * 80)

print(f"Train records      : {len(train_df)}")
print(f"Validation records : {len(val_df)}")
print("Test records loaded: NO")


# ============================================================
# 5. IMAGE NORMALIZATION
# ============================================================

# Stored NPY format:
#
#     float32
#     shape = (224, 224, 3)
#     range = [-1, +1]
#
# We first convert:
#
#     [-1,1] -> [0,1]
#
# Then apply ImageNet normalization:
#
#     (x - mean) / std
#
# This is appropriate for ImageNet-pretrained CoAtNet.

IMAGENET_MEAN = (
    0.485,
    0.456,
    0.406
)

IMAGENET_STD = (
    0.229,
    0.224,
    0.225
)


# ============================================================
# 6. TRAINING AUGMENTATION
# ============================================================

# Important:
# No horizontal flip.
# No vertical flip.
# No random crop.
# No elastic deformation.
# No MixUp.
# No CutMix.
# No RandomErasing.
#
# We keep anatomical structure stable.

train_transform = transforms.Compose([
    
    # NPY is already [-1,1].
    # Convert to [0,1] before torchvision transforms.
    transforms.Lambda(
        lambda x: torch.clamp(
            (x + 1.0) / 2.0,
            0.0,
            1.0
        )
    ),

    # Mild geometric variation.
    transforms.RandomAffine(
        degrees=5,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05),
        shear=0,
        interpolation=transforms.InterpolationMode.BILINEAR,
        fill=0
    ),

    # Mild intensity variation.
    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08,
        saturation=0.0,
        hue=0.0
    ),

    # ImageNet-compatible normalization.
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ============================================================
# 7. VALIDATION TRANSFORMATION
# ============================================================

# Absolutely NO random augmentation during validation.

val_transform = transforms.Compose([

    # [-1,1] -> [0,1]
    transforms.Lambda(
        lambda x: torch.clamp(
            (x + 1.0) / 2.0,
            0.0,
            1.0
        )
    ),

    # ImageNet normalization.
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# ============================================================
# 8. FINAL DATASET CLASS
# ============================================================

class FinalCoAtNetRun2Dataset(Dataset):

    def __init__(
        self,
        dataframe,
        input_root,
        transform=None
    ):

        self.df = dataframe.reset_index(drop=True).copy()
        self.input_root = Path(input_root)
        self.transform = transform

        self.condition_ids = (
            self.df[CONDITION_COL]
            .astype(str)
            .tolist()
        )

        self.labels = (
            self.df[TARGET_COL]
            .astype(int)
            .tolist()
        )

        # ----------------------------------------------------
        # Verify every physical input before training.
        # ----------------------------------------------------

        missing = []

        for condition_id in self.condition_ids:

            npy_path = (
                self.input_root
                / f"{condition_id}.npy"
            )

            if not npy_path.exists():
                missing.append(condition_id)

        if len(missing) > 0:

            raise FileNotFoundError(
                f"Missing {len(missing)} NPY files. "
                f"First examples: {missing[:10]}"
            )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        condition_id = self.condition_ids[index]
        label = self.labels[index]

        npy_path = (
            self.input_root
            / f"{condition_id}.npy"
        )

        # ----------------------------------------------------
        # Load stored image
        # ----------------------------------------------------

        image = np.load(
            npy_path,
            allow_pickle=False
        )

        image = image.astype(
            np.float32,
            copy=False
        )

        # ----------------------------------------------------
        # HARD INPUT VALIDATION
        # ----------------------------------------------------

        if image.shape != (224, 224, 3):

            raise ValueError(
                f"Invalid shape for {condition_id}: "
                f"{image.shape}"
            )

        if not np.isfinite(image).all():

            raise ValueError(
                f"Non-finite values detected: "
                f"{condition_id}"
            )

        min_value = float(image.min())
        max_value = float(image.max())

        if min_value < -1.0001 or max_value > 1.0001:

            raise ValueError(
                f"Unexpected NPY range for "
                f"{condition_id}: "
                f"[{min_value}, {max_value}]"
            )

        # ----------------------------------------------------
        # Convert numpy -> torch
        #
        # HWC -> CHW
        # ----------------------------------------------------

        tensor = torch.from_numpy(
            image
        ).permute(
            2, 0, 1
        )

        # ----------------------------------------------------
        # Apply transforms
        # ----------------------------------------------------

        if self.transform is not None:

            tensor = self.transform(
                tensor
            )

        # ----------------------------------------------------
        # Final tensor validation
        # ----------------------------------------------------

        if tensor.shape != (3, 224, 224):

            raise ValueError(
                f"Invalid final tensor shape "
                f"for {condition_id}: "
                f"{tuple(tensor.shape)}"
            )

        if not torch.isfinite(tensor).all():

            raise ValueError(
                f"Non-finite final tensor: "
                f"{condition_id}"
            )

        return (
            tensor,
            torch.tensor(
                label,
                dtype=torch.long
            ),
            condition_id
        )


# ============================================================
# 9. CREATE DATASETS
# ============================================================

train_dataset = FinalCoAtNetRun2Dataset(
    dataframe=train_df,
    input_root=INPUT_ROOT,
    transform=train_transform
)

val_dataset = FinalCoAtNetRun2Dataset(
    dataframe=val_df,
    input_root=INPUT_ROOT,
    transform=val_transform
)

print("\nDATASETS")
print("-" * 80)

print(f"Train dataset      : {len(train_dataset)}")
print(f"Validation dataset : {len(val_dataset)}")


# ============================================================
# 10. CLASS DISTRIBUTION
# ============================================================

train_counts = (
    train_df[TARGET_COL]
    .value_counts()
    .sort_index()
)

ds_count = int(
    train_counts.get(0, 0)
)

dr_count = int(
    train_counts.get(1, 0)
)

print("\nTRAIN CLASS DISTRIBUTION")
print("-" * 80)

print(f"DS-TB : {ds_count}")
print(f"DR-TB : {dr_count}")


# ============================================================
# 11. CLASS WEIGHTS
# ============================================================

# Balanced class weighting:
#
# weight_class = N / (number_of_classes * class_count)

num_classes = 2
total_train = ds_count + dr_count

class_weights = torch.tensor(
    [
        total_train / (num_classes * ds_count),
        total_train / (num_classes * dr_count)
    ],
    dtype=torch.float32
)

print("\nCLASS WEIGHTS")
print("-" * 80)

print(
    f"DS-TB weight : "
    f"{class_weights[0].item():.6f}"
)

print(
    f"DR-TB weight : "
    f"{class_weights[1].item():.6f}"
)


# ============================================================
# 12. DATA LOADERS
# ============================================================

BATCH_SIZE = 4
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    drop_last=False
)

print("\nDATALOADERS")
print("-" * 80)

print(
    f"Train batches      : "
    f"{len(train_loader)}"
)

print(
    f"Validation batches : "
    f"{len(val_loader)}"
)

print(
    f"Batch size         : "
    f"{BATCH_SIZE}"
)

print(
    f"Workers            : "
    f"{NUM_WORKERS}"
)

print(
    "Shuffle train      : YES"
)

print(
    "Shuffle validation : NO"
)


# ============================================================
# 13. SAMPLE QC
# ============================================================

print("\nSAMPLE TENSOR QC")
print("-" * 80)

train_sample_image, train_sample_label, train_sample_id = (
    train_dataset[0]
)

val_sample_image, val_sample_label, val_sample_id = (
    val_dataset[0]
)

print(
    "Train sample:"
)

print(
    f"  condition : {train_sample_id}"
)

print(
    f"  label     : {train_sample_label.item()}"
)

print(
    f"  shape     : {tuple(train_sample_image.shape)}"
)

print(
    f"  dtype     : {train_sample_image.dtype}"
)

print(
    f"  min       : {train_sample_image.min().item():.6f}"
)

print(
    f"  max       : {train_sample_image.max().item():.6f}"
)

print(
    f"  finite    : {bool(torch.isfinite(train_sample_image).all())}"
)


print(
    "\nValidation sample:"
)

print(
    f"  condition : {val_sample_id}"
)

print(
    f"  label     : {val_sample_label.item()}"
)

print(
    f"  shape     : {tuple(val_sample_image.shape)}"
)

print(
    f"  dtype     : {val_sample_image.dtype}"
)

print(
    f"  min       : {val_sample_image.min().item():.6f}"
)

print(
    f"  max       : {val_sample_image.max().item():.6f}"
)

print(
    f"  finite    : {bool(torch.isfinite(val_sample_image).all())}"
)


# ============================================================
# 14. BATCH QC
# ============================================================

print("\nBATCH QC")
print("-" * 80)

batch_images, batch_labels, batch_ids = next(
    iter(train_loader)
)

print(
    f"Image batch shape : "
    f"{tuple(batch_images.shape)}"
)

print(
    f"Label batch shape : "
    f"{tuple(batch_labels.shape)}"
)

print(
    f"Image dtype       : "
    f"{batch_images.dtype}"
)

print(
    f"Image finite      : "
    f"{bool(torch.isfinite(batch_images).all())}"
)

print(
    f"Labels            : "
    f"{batch_labels.tolist()}"
)

print(
    f"Condition IDs     : "
    f"{list(batch_ids)}"
)


# ============================================================
# 15. IMPORTANT NORMALIZATION CHECK
# ============================================================

# After ImageNet normalization, the range does NOT have
# to be [-1,1].
#
# Therefore we only verify finiteness here.
#
# This is intentional.

assert torch.isfinite(batch_images).all()

assert batch_images.shape == (
    BATCH_SIZE,
    3,
    224,
    224
)

assert batch_labels.shape == (
    BATCH_SIZE,
)


# ============================================================
# 16. SAVE DATASET CONFIGURATION
# ============================================================

dataset_config = {

    "notebook":
        "10C_CXR_CoAtNet_Final_Training_Run2",

    "cell":
        "Cell 2",

    "train_records":
        len(train_dataset),

    "validation_records":
        len(val_dataset),

    "batch_size":
        BATCH_SIZE,

    "num_workers":
        NUM_WORKERS,

    "stored_input_range":
        "[-1,1]",

    "runtime_input_conversion":
        "[-1,1] -> [0,1]",

    "normalization":
        "ImageNet",

    "imagenet_mean":
        list(IMAGENET_MEAN),

    "imagenet_std":
        list(IMAGENET_STD),

    "train_augmentation": [
        "RandomAffine(degrees=5)",
        "translate=(0.05,0.05)",
        "scale=(0.95,1.05)",
        "ColorJitter(brightness=0.08,contrast=0.08)"
    ],

    "validation_augmentation":
        "None",

    "sampler":
        "Standard shuffle",

    "class_weighted_loss":
        True,

    "horizontal_flip":
        False,

    "vertical_flip":
        False,

    "random_crop":
        False,

    "elastic_deformation":
        False,

    "mixup":
        False,

    "cutmix":
        False,

    "random_erasing":
        False,

    "test_loaded":
        False,

    "test_used":
        False
}

dataset_config_path = (
    OUTPUT_ROOT
    / "Notebook10C_Cell2_Run2_Dataset_Config.json"
)

with open(
    dataset_config_path,
    "w"
) as f:

    json.dump(
        dataset_config,
        f,
        indent=2
    )


# ============================================================
# 17. FINAL SAFETY ASSERTIONS
# ============================================================

assert len(train_dataset) == 1185
assert len(val_dataset) == 395

assert ds_count == 422
assert dr_count == 763

assert class_weights.shape == (2,)

assert class_weights[0] > class_weights[1]

assert len(train_loader) == 297
assert len(val_loader) == 99


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 80)
print("CELL 2 FINAL SAFETY GATE")
print("=" * 80)

print("Train dataset             : PASS")
print("Validation dataset        : PASS")
print("Test dataset loaded       : NO")
print("Image shape               : PASS")
print("Stored range              : PASS")
print("ImageNet normalization    : PASS")
print("Training augmentation     : PASS")
print("Validation no augmentation: PASS")
print("Class weighting           : PASS")
print("DataLoader                : PASS")
print("Tensor finite check       : PASS")
print("Test isolation            : PASS")

print("\n" + "=" * 80)
print("CELL 2 STATUS: PASS")
print("=" * 80)

NOTEBOOK 10C — CELL 2
FINAL RUN-2 DATASET + AUGMENTATION PIPELINE
Device : cpu

SPLIT CHECK
--------------------------------------------------------------------------------
Train records      : 1185
Validation records : 395
Test records loaded: NO

DATASETS
--------------------------------------------------------------------------------
Train dataset      : 1185
Validation dataset : 395

TRAIN CLASS DISTRIBUTION
--------------------------------------------------------------------------------
DS-TB : 422
DR-TB : 763

CLASS WEIGHTS
--------------------------------------------------------------------------------
DS-TB weight : 1.404028
DR-TB weight : 0.776540

DATALOADERS
--------------------------------------------------------------------------------
Train batches      : 297
Validation batches : 99
Batch size         : 4
Workers            : 0
Shuffle train      : YES
Shuffle validation : NO

SAMPLE TENSOR QC
-------------------------------------------------------------------------------

In [3]:
# ============================================================
# NOTEBOOK 10C
# CXR CoAtNet FINAL TRAINING — RUN 2
#
# CELL 3 — MODEL + OPTIMIZER + LOSS + SCHEDULER
# ============================================================

import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import timm

from pathlib import Path


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("NOTEBOOK 10C — CELL 3")
print("CoAtNet-0 RUN-2 MODEL CONFIGURATION")
print("=" * 80)

print(f"Device : {DEVICE}")


# ============================================================
# 3. OUTPUT PATH
# ============================================================

BASE_ROOT = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation"
)

FINAL_COHORT_ROOT = (
    BASE_ROOT
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

OUTPUT_ROOT = (
    FINAL_COHORT_ROOT
    / "CoAtNet_Training_Run2_10C"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 4. TRAINING HYPERPARAMETERS
# ============================================================

NUM_CLASSES = 2

IMAGE_SIZE = 224

BATCH_SIZE = 4

GRADIENT_ACCUMULATION_STEPS = 4

EFFECTIVE_BATCH_SIZE = (
    BATCH_SIZE *
    GRADIENT_ACCUMULATION_STEPS
)

NUM_EPOCHS = 40

# ------------------------------------------------------------
# Differential learning rates
#
# Backbone receives a smaller learning rate because it starts
# from ImageNet pretrained weights.
#
# Classification head receives a larger learning rate.
# ------------------------------------------------------------

BACKBONE_LR = 1e-5

CLASSIFIER_LR = 1e-4

WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.02

MAX_GRAD_NORM = 1.0

MIN_LR = 1e-7


# ============================================================
# 5. CREATE COATNET
# ============================================================

print("\nCREATING COATNET-0")
print("-" * 80)

model = timm.create_model(
    "coatnet_0_rw_224",
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(DEVICE)


# ============================================================
# 6. MODEL PARAMETER INSPECTION
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Model class       : "
    f"{model.__class__.__name__}"
)

print(
    f"Total parameters  : "
    f"{total_params:,}"
)

print(
    f"Trainable params  : "
    f"{trainable_params:,}"
)


# ============================================================
# 7. VERIFY CLASSIFIER
# ============================================================

print("\nCLASSIFIER")
print("-" * 80)

print(model.head)

classifier_parameters = list(
    model.head.parameters()
)

classifier_parameter_ids = {
    id(p)
    for p in classifier_parameters
}

classifier_parameter_count = sum(
    p.numel()
    for p in classifier_parameters
)

print(
    f"Classifier parameters : "
    f"{classifier_parameter_count:,}"
)


# ============================================================
# 8. BACKBONE PARAMETERS
# ============================================================

backbone_parameters = [
    p
    for p in model.parameters()
    if id(p) not in classifier_parameter_ids
]

backbone_parameter_count = sum(
    p.numel()
    for p in backbone_parameters
)

print(
    f"Backbone parameters   : "
    f"{backbone_parameter_count:,}"
)


# ============================================================
# 9. VERIFY ALL PARAMETERS ARE TRAINABLE
# ============================================================

frozen_parameters = [
    name
    for name, p in model.named_parameters()
    if not p.requires_grad
]

print(
    f"Frozen parameters     : "
    f"{len(frozen_parameters)}"
)

assert len(frozen_parameters) == 0


# ============================================================
# 10. PARAMETER GROUPS
# ============================================================

optimizer = torch.optim.AdamW(
    [
        {
            "params": backbone_parameters,
            "lr": BACKBONE_LR,
            "weight_decay": WEIGHT_DECAY,
            "name": "backbone"
        },
        {
            "params": classifier_parameters,
            "lr": CLASSIFIER_LR,
            "weight_decay": WEIGHT_DECAY,
            "name": "classifier"
        }
    ]
)


# ============================================================
# 11. CLASS-WEIGHTED CROSS ENTROPY
# ============================================================

# Values obtained from Cell 2:
#
# DS = 422
# DR = 763
#
# Balanced class weights:
#
# DS = 1.404028
# DR = 0.776540

CLASS_WEIGHTS = torch.tensor(
    [
        1.404028,
        0.776540
    ],
    dtype=torch.float32,
    device=DEVICE
)

criterion = nn.CrossEntropyLoss(
    weight=CLASS_WEIGHTS,
    label_smoothing=LABEL_SMOOTHING
)


# ============================================================
# 12. COSINE ANNEALING
# ============================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=MIN_LR
)


# ============================================================
# 13. MODEL FORWARD SANITY TEST
# ============================================================

print("\nFORWARD-PASS SANITY TEST")
print("-" * 80)

model.eval()

with torch.no_grad():

    dummy_input = torch.randn(
        2,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        dtype=torch.float32,
        device=DEVICE
    )

    dummy_output = model(
        dummy_input
    )

print(
    f"Input shape  : "
    f"{tuple(dummy_input.shape)}"
)

print(
    f"Output shape : "
    f"{tuple(dummy_output.shape)}"
)

assert dummy_output.shape == (
    2,
    NUM_CLASSES
)

assert torch.isfinite(
    dummy_output
).all()


# ============================================================
# 14. SOFTMAX SANITY TEST
# ============================================================

dummy_probabilities = torch.softmax(
    dummy_output,
    dim=1
)

probability_sums = (
    dummy_probabilities.sum(dim=1)
)

print(
    "Probability sums :",
    probability_sums.detach().cpu().numpy()
)

assert torch.allclose(
    probability_sums,
    torch.ones_like(probability_sums),
    atol=1e-5
)


# ============================================================
# 15. GRADIENT SANITY TEST
# ============================================================

print("\nGRADIENT SANITY TEST")
print("-" * 80)

model.train()

test_input = torch.randn(
    2,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
    dtype=torch.float32,
    device=DEVICE
)

test_labels = torch.tensor(
    [0, 1],
    dtype=torch.long,
    device=DEVICE
)

test_output = model(
    test_input
)

test_loss = criterion(
    test_output,
    test_labels
)

optimizer.zero_grad(
    set_to_none=True
)

test_loss.backward()

nonzero_gradients = 0
total_gradient_tensors = 0

for parameter in model.parameters():

    if parameter.requires_grad:

        total_gradient_tensors += 1

        if (
            parameter.grad is not None
            and
            torch.isfinite(parameter.grad).all()
            and
            parameter.grad.abs().sum().item() > 0
        ):
            nonzero_gradients += 1


optimizer.zero_grad(
    set_to_none=True
)

print(
    f"Gradient tensors      : "
    f"{total_gradient_tensors}"
)

print(
    f"Non-zero gradients    : "
    f"{nonzero_gradients}"
)

assert total_gradient_tensors > 0

assert nonzero_gradients > 0


# ============================================================
# 16. LEARNING-RATE SANITY CHECK
# ============================================================

print("\nLEARNING RATE CONFIGURATION")
print("-" * 80)

for group in optimizer.param_groups:

    print(
        f"{group['name']:<12} "
        f"LR={group['lr']:.2e} "
        f"WD={group['weight_decay']:.2e}"
    )

assert (
    optimizer.param_groups[0]["lr"]
    == BACKBONE_LR
)

assert (
    optimizer.param_groups[1]["lr"]
    == CLASSIFIER_LR
)


# ============================================================
# 17. SAVE CONFIGURATION
# ============================================================

config = {

    "notebook":
        "10C_CXR_CoAtNet_Final_Training_Run2",

    "cell":
        "Cell 3",

    "model":
        "coatnet_0_rw_224",

    "pretrained":
        True,

    "num_classes":
        NUM_CLASSES,

    "image_size":
        IMAGE_SIZE,

    "total_parameters":
        int(total_params),

    "trainable_parameters":
        int(trainable_params),

    "backbone_parameters":
        int(backbone_parameter_count),

    "classifier_parameters":
        int(classifier_parameter_count),

    "backbone_learning_rate":
        BACKBONE_LR,

    "classifier_learning_rate":
        CLASSIFIER_LR,

    "weight_decay":
        WEIGHT_DECAY,

    "label_smoothing":
        LABEL_SMOOTHING,

    "class_weights":
        CLASS_WEIGHTS.detach().cpu().tolist(),

    "optimizer":
        "AdamW",

    "scheduler":
        "CosineAnnealingLR",

    "epochs":
        NUM_EPOCHS,

    "minimum_learning_rate":
        MIN_LR,

    "batch_size":
        BATCH_SIZE,

    "gradient_accumulation_steps":
        GRADIENT_ACCUMULATION_STEPS,

    "effective_batch_size":
        EFFECTIVE_BATCH_SIZE,

    "max_gradient_norm":
        MAX_GRAD_NORM,

    "early_stopping":
        False,

    "test_used":
        False,

    "validation_used_for_model_selection":
        True
}

config_path = (
    OUTPUT_ROOT
    / "Notebook10C_Cell3_Run2_Model_Config.json"
)

with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# ============================================================
# 18. FINAL SAFETY GATE
# ============================================================

print("\n" + "=" * 80)
print("CELL 3 FINAL SAFETY GATE")
print("=" * 80)

print("CoAtNet-0 loaded                 : PASS")
print("ImageNet pretrained weights      : PASS")
print("All parameters trainable         : PASS")
print("Backbone differential LR         : PASS")
print("Classifier differential LR       : PASS")
print("Class-weighted loss              : PASS")
print("Label smoothing                  : PASS")
print("AdamW                            : PASS")
print("Cosine scheduler                 : PASS")
print("Gradient sanity                  : PASS")
print("Forward-pass sanity              : PASS")
print("Early stopping                   : DISABLED")
print("Test set used                    : NO")

print("\n" + "=" * 80)
print("CELL 3 STATUS: PASS")
print("=" * 80)

NOTEBOOK 10C — CELL 3
CoAtNet-0 RUN-2 MODEL CONFIGURATION
Device : cpu

CREATING COATNET-0
--------------------------------------------------------------------------------
Model class       : MaxxVit
Total parameters  : 26,668,100
Trainable params  : 26,668,100

CLASSIFIER
--------------------------------------------------------------------------------
ClassifierHead(
  (global_pool): SelectAdaptivePool2d(pool_type=avg, flatten=Flatten(start_dim=1, end_dim=-1))
  (drop): Dropout(p=0.0, inplace=False)
  (fc): Linear(in_features=768, out_features=2, bias=True)
  (flatten): Identity()
)
Classifier parameters : 1,538
Backbone parameters   : 26,666,562
Frozen parameters     : 0

FORWARD-PASS SANITY TEST
--------------------------------------------------------------------------------
Input shape  : (2, 3, 224, 224)
Output shape : (2, 2)
Probability sums : [1. 1.]

GRADIENT SANITY TEST
--------------------------------------------------------------------------------
Gradient tensors      : 194

In [4]:
# ============================================================
# NOTEBOOK 10C
# CXR CoAtNet FINAL TRAINING — RUN 2
#
# CELL 4 — FULL 40-EPOCH TRAINING
#
# IMPORTANT:
# - Test set is NEVER loaded
# - No early stopping
# - Exactly 40 epochs
# - Validation used only for checkpoint selection
# - Best checkpoint selected by validation ROC-AUC
# - >90% is a target, NOT artificially enforced
# ============================================================

import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("NOTEBOOK 10C — CELL 4")
print("CoAtNet-0 RUN-2 FULL TRAINING")
print("=" * 80)

print(f"Device : {DEVICE}")


# ============================================================
# 3. PATHS
# ============================================================

BASE_ROOT = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation"
)

FINAL_COHORT_ROOT = (
    BASE_ROOT
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

OUTPUT_ROOT = (
    FINAL_COHORT_ROOT
    / "CoAtNet_Training_Run2_10C"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_ROOT = (
    OUTPUT_ROOT
    / "checkpoints"
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

LOG_ROOT = (
    OUTPUT_ROOT
    / "logs"
)

LOG_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / "Notebook10C_Run2_Best_Validation_ROC_AUC.pth"
)

FINAL_CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / "Notebook10C_Run2_Final_Epoch40.pth"
)

TRAINING_LOG_PATH = (
    LOG_ROOT
    / "Notebook10C_Run2_Training_Log.csv"
)

TARGET_HISTORY_PATH = (
    LOG_ROOT
    / "Notebook10C_Run2_Target_Tracking.csv"
)


# ============================================================
# 4. VERIFY CELL 2 OBJECTS
# ============================================================

required_objects = {
    "train_loader": "train_loader",
    "val_loader": "val_loader",
    "train_df": "train_df",
    "val_df": "val_df",
}

print("\nOBJECT VERIFICATION")
print("-" * 80)

for name, variable_name in required_objects.items():

    exists = variable_name in globals()

    print(
        f"{name:<20}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    assert exists


# ============================================================
# 5. VERIFY CELL 3 OBJECTS
# ============================================================

required_model_objects = {
    "model": "model",
    "optimizer": "optimizer",
    "criterion": "criterion",
    "scheduler": "scheduler",
}

print("\nMODEL OBJECT VERIFICATION")
print("-" * 80)

for name, variable_name in required_model_objects.items():

    exists = variable_name in globals()

    print(
        f"{name:<20}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    assert exists


# ============================================================
# 6. TRAINING CONSTANTS
# ============================================================

NUM_EPOCHS = 40

GRADIENT_ACCUMULATION_STEPS = 4

MAX_GRAD_NORM = 1.0

TARGET_TRAIN_ACCURACY = 0.90

TARGET_VAL_ACCURACY = 0.90

TARGET_VAL_ROC_AUC = 0.90


# ============================================================
# 7. IMPORTANT — TEST ISOLATION
# ============================================================

# We deliberately DO NOT load Notebook08_Cell25_TEST.csv.
#
# This cell only references:
#
#     train_loader
#     val_loader
#
# Therefore the test set cannot influence training.

print("\nTEST ISOLATION")
print("-" * 80)

print("Test dataframe loaded       : NO")
print("Test DataLoader created     : NO")
print("Test predictions generated  : NO")
print("Test accuracy calculated    : NO")
print("Test used for checkpointing : NO")
print("Test used for thresholding  : NO")


# ============================================================
# 8. METRIC FUNCTION
# ============================================================

def calculate_metrics(
    labels,
    probabilities,
    predictions
):

    labels = np.asarray(
        labels,
        dtype=np.int64
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    predictions = np.asarray(
        predictions,
        dtype=np.int64
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    balanced_accuracy = balanced_accuracy_score(
        labels,
        predictions
    )

    precision = precision_score(
        labels,
        predictions,
        zero_division=0
    )

    sensitivity = recall_score(
        labels,
        predictions,
        zero_division=0
    )

    specificity = recall_score(
        labels,
        predictions,
        pos_label=0,
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        zero_division=0
    )

    try:

        roc_auc = roc_auc_score(
            labels,
            probabilities
        )

    except ValueError:

        roc_auc = np.nan

    try:

        pr_auc = average_precision_score(
            labels,
            probabilities
        )

    except ValueError:

        pr_auc = np.nan

    cm = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    return {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(
            balanced_accuracy
        ),
        "precision": float(precision),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "f1": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


# ============================================================
# 9. TRAINING FUNCTION
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    accumulation_steps,
    max_grad_norm
):

    model.train()

    running_loss = 0.0

    all_labels = []
    all_probabilities = []
    all_predictions = []

    optimizer.zero_grad(
        set_to_none=True
    )

    optimizer_steps = 0

    for batch_index, batch in enumerate(loader):

        images, labels, condition_ids = batch

        images = images.to(
            device,
            non_blocking=False
        )

        labels = labels.to(
            device,
            non_blocking=False
        )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        logits = model(images)

        loss = criterion(
            logits,
            labels
        )

        # ----------------------------------------------------
        # Gradient accumulation
        # ----------------------------------------------------

        scaled_loss = (
            loss /
            accumulation_steps
        )

        scaled_loss.backward()

        # ----------------------------------------------------
        # Optimizer step
        # ----------------------------------------------------

        should_step = (
            ((batch_index + 1) % accumulation_steps == 0)
            or
            ((batch_index + 1) == len(loader))
        )

        if should_step:

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_grad_norm
            )

            optimizer.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            optimizer_steps += 1

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        probabilities = torch.softmax(
            logits.detach(),
            dim=1
        )[:, 1]

        predictions = (
            probabilities >= 0.5
        ).long()

        running_loss += (
            loss.item() *
            labels.size(0)
        )

        all_labels.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_probabilities.extend(
            probabilities.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    epoch_loss = (
        running_loss /
        len(loader.dataset)
    )

    metrics = calculate_metrics(
        all_labels,
        all_probabilities,
        all_predictions
    )

    metrics["loss"] = float(
        epoch_loss
    )

    metrics["optimizer_steps"] = int(
        optimizer_steps
    )

    return metrics


# ============================================================
# 10. VALIDATION FUNCTION
# ============================================================

@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_probabilities = []
    all_predictions = []

    for batch in loader:

        images, labels, condition_ids = batch

        images = images.to(
            device,
            non_blocking=False
        )

        labels = labels.to(
            device,
            non_blocking=False
        )

        logits = model(images)

        loss = criterion(
            logits,
            labels
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        predictions = (
            probabilities >= 0.5
        ).long()

        running_loss += (
            loss.item() *
            labels.size(0)
        )

        all_labels.extend(
            labels.cpu()
            .numpy()
            .tolist()
        )

        all_probabilities.extend(
            probabilities.cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions.cpu()
            .numpy()
            .tolist()
        )

    epoch_loss = (
        running_loss /
        len(loader.dataset)
    )

    metrics = calculate_metrics(
        all_labels,
        all_probabilities,
        all_predictions
    )

    metrics["loss"] = float(
        epoch_loss
    )

    return metrics


# ============================================================
# 11. BEST-MODEL TRACKING
# ============================================================

best_val_roc_auc = -np.inf

best_val_accuracy = -np.inf

best_epoch = None

history = []

target_history = []


# ============================================================
# 12. TRAINING START
# ============================================================

print("\n" + "=" * 80)
print("STARTING RUN-2 TRAINING")
print("=" * 80)

print(
    f"Total epochs             : {NUM_EPOCHS}"
)

print(
    f"Gradient accumulation    : "
    f"{GRADIENT_ACCUMULATION_STEPS}"
)

print(
    f"Effective batch size     : "
    f"{4 * GRADIENT_ACCUMULATION_STEPS}"
)

print(
    f"Backbone LR              : "
    f"{optimizer.param_groups[0]['lr']:.8e}"
)

print(
    f"Classifier LR            : "
    f"{optimizer.param_groups[1]['lr']:.8e}"
)

print(
    f"Early stopping           : DISABLED"
)

print(
    f"Test set                 : NOT USED"
)

print(
    f"Target train accuracy    : >90%"
)

print(
    f"Target validation acc.   : >90%"
)

print(
    f"Target validation AUC    : >90%"
)

print("=" * 80)


# ============================================================
# 13. EPOCH LOOP
# ============================================================

training_start = time.time()

for epoch in range(
    1,
    NUM_EPOCHS + 1
):

    epoch_start = time.time()

    print("\n")
    print("=" * 80)

    print(
        f"EPOCH {epoch}/{NUM_EPOCHS}"
    )

    print("=" * 80)

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE,
        accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        max_grad_norm=MAX_GRAD_NORM
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    val_metrics = validate_one_epoch(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE
    )

    # --------------------------------------------------------
    # LEARNING RATES
    # --------------------------------------------------------

    backbone_lr = (
        optimizer.param_groups[0]["lr"]
    )

    classifier_lr = (
        optimizer.param_groups[1]["lr"]
    )

    # --------------------------------------------------------
    # EPOCH TIME
    # --------------------------------------------------------

    epoch_time = (
        time.time() -
        epoch_start
    )

    # --------------------------------------------------------
    # TARGET CHECK
    # --------------------------------------------------------

    train_target_pass = (
        train_metrics["accuracy"]
        > TARGET_TRAIN_ACCURACY
    )

    val_accuracy_target_pass = (
        val_metrics["accuracy"]
        > TARGET_VAL_ACCURACY
    )

    val_auc_target_pass = (
        val_metrics["roc_auc"]
        > TARGET_VAL_ROC_AUC
    )

    all_development_targets_pass = (
        train_target_pass
        and
        val_accuracy_target_pass
        and
        val_auc_target_pass
    )

    # --------------------------------------------------------
    # BEST CHECKPOINT
    # --------------------------------------------------------

    current_val_auc = (
        val_metrics["roc_auc"]
    )

    is_best = (
        current_val_auc >
        best_val_roc_auc
    )

    if is_best:

        best_val_roc_auc = (
            current_val_auc
        )

        best_val_accuracy = (
            val_metrics["accuracy"]
        )

        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "scheduler_state_dict":
                    scheduler.state_dict(),

                "best_val_roc_auc":
                    best_val_roc_auc,

                "best_val_accuracy":
                    best_val_accuracy,

                "train_metrics":
                    train_metrics,

                "val_metrics":
                    val_metrics,

                "seed": SEED,

                "model_name":
                    "coatnet_0_rw_224",

                "test_used":
                    False,

                "early_stopping":
                    False
            },
            BEST_CHECKPOINT_PATH
        )

    # --------------------------------------------------------
    # TRAINING HISTORY
    # --------------------------------------------------------

    row = {

        "epoch": epoch,

        "train_loss":
            train_metrics["loss"],

        "train_accuracy":
            train_metrics["accuracy"],

        "train_balanced_accuracy":
            train_metrics["balanced_accuracy"],

        "train_precision":
            train_metrics["precision"],

        "train_sensitivity":
            train_metrics["sensitivity"],

        "train_specificity":
            train_metrics["specificity"],

        "train_f1":
            train_metrics["f1"],

        "train_roc_auc":
            train_metrics["roc_auc"],

        "train_pr_auc":
            train_metrics["pr_auc"],

        "val_loss":
            val_metrics["loss"],

        "val_accuracy":
            val_metrics["accuracy"],

        "val_balanced_accuracy":
            val_metrics["balanced_accuracy"],

        "val_precision":
            val_metrics["precision"],

        "val_sensitivity":
            val_metrics["sensitivity"],

        "val_specificity":
            val_metrics["specificity"],

        "val_f1":
            val_metrics["f1"],

        "val_roc_auc":
            val_metrics["roc_auc"],

        "val_pr_auc":
            val_metrics["pr_auc"],

        "val_tn":
            val_metrics["tn"],

        "val_fp":
            val_metrics["fp"],

        "val_fn":
            val_metrics["fn"],

        "val_tp":
            val_metrics["tp"],

        "backbone_lr":
            backbone_lr,

        "classifier_lr":
            classifier_lr,

        "optimizer_steps":
            train_metrics["optimizer_steps"],

        "epoch_time_seconds":
            epoch_time,

        "best_so_far":
            is_best,

        "best_val_roc_auc":
            best_val_roc_auc,

        "best_epoch":
            best_epoch,

        "train_target_gt_90":
            train_target_pass,

        "val_accuracy_target_gt_90":
            val_accuracy_target_pass,

        "val_auc_target_gt_90":
            val_auc_target_pass,

        "all_development_targets_gt_90":
            all_development_targets_pass
    }

    history.append(row)

    # --------------------------------------------------------
    # TARGET HISTORY
    # --------------------------------------------------------

    target_row = {

        "epoch": epoch,

        "train_accuracy":
            train_metrics["accuracy"],

        "validation_accuracy":
            val_metrics["accuracy"],

        "validation_roc_auc":
            val_metrics["roc_auc"],

        "train_gt_90":
            train_target_pass,

        "validation_accuracy_gt_90":
            val_accuracy_target_pass,

        "validation_auc_gt_90":
            val_auc_target_pass,

        "all_targets_gt_90":
            all_development_targets_pass
    }

    target_history.append(
        target_row
    )

    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print(
        f"\nTrain Loss       : "
        f"{train_metrics['loss']:.5f}"
    )

    print(
        f"Train Accuracy   : "
        f"{train_metrics['accuracy']:.5f} "
        f"({train_metrics['accuracy'] * 100:.2f}%)"
    )

    print(
        f"Train Balanced   : "
        f"{train_metrics['balanced_accuracy']:.5f}"
    )

    print(
        f"Train F1         : "
        f"{train_metrics['f1']:.5f}"
    )

    print(
        f"Train ROC-AUC    : "
        f"{train_metrics['roc_auc']:.5f}"
    )

    print(
        f"Train PR-AUC     : "
        f"{train_metrics['pr_auc']:.5f}"
    )

    print(
        f"\nVal Loss         : "
        f"{val_metrics['loss']:.5f}"
    )

    print(
        f"Val Accuracy     : "
        f"{val_metrics['accuracy']:.5f} "
        f"({val_metrics['accuracy'] * 100:.2f}%)"
    )

    print(
        f"Val Balanced Acc : "
        f"{val_metrics['balanced_accuracy']:.5f}"
    )

    print(
        f"Val Precision    : "
        f"{val_metrics['precision']:.5f}"
    )

    print(
        f"Val Sensitivity  : "
        f"{val_metrics['sensitivity']:.5f}"
    )

    print(
        f"Val Specificity  : "
        f"{val_metrics['specificity']:.5f}"
    )

    print(
        f"Val F1           : "
        f"{val_metrics['f1']:.5f}"
    )

    print(
        f"Val ROC-AUC      : "
        f"{val_metrics['roc_auc']:.5f}"
    )

    print(
        f"Val PR-AUC       : "
        f"{val_metrics['pr_auc']:.5f}"
    )

    print(
        f"\nVal CM           : "
        f"TN={val_metrics['tn']} "
        f"FP={val_metrics['fp']} "
        f"FN={val_metrics['fn']} "
        f"TP={val_metrics['tp']}"
    )

    print(
        f"\nBackbone LR      : "
        f"{backbone_lr:.8e}"
    )

    print(
        f"Classifier LR    : "
        f"{classifier_lr:.8e}"
    )

    print(
        f"Epoch Time       : "
        f"{epoch_time:.2f} sec"
    )

    print(
        f"\n>90% TARGET STATUS"
    )

    print(
        f"Train Accuracy   : "
        f"{'PASS' if train_target_pass else 'NOT YET'}"
    )

    print(
        f"Val Accuracy     : "
        f"{'PASS' if val_accuracy_target_pass else 'NOT YET'}"
    )

    print(
        f"Val ROC-AUC      : "
        f"{'PASS' if val_auc_target_pass else 'NOT YET'}"
    )

    if is_best:

        print(
            "\n*** NEW BEST VALIDATION ROC-AUC ***"
        )

    # --------------------------------------------------------
    # SCHEDULER
    # --------------------------------------------------------

    scheduler.step()

    # --------------------------------------------------------
    # SAVE CURRENT LOG
    # --------------------------------------------------------

    pd.DataFrame(history).to_csv(
        TRAINING_LOG_PATH,
        index=False
    )

    pd.DataFrame(target_history).to_csv(
        TARGET_HISTORY_PATH,
        index=False
    )


# ============================================================
# 14. FINAL CHECKPOINT
# ============================================================

total_training_time = (
    time.time() -
    training_start
)

torch.save(
    {
        "epoch": NUM_EPOCHS,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_epoch":
            best_epoch,

        "best_val_roc_auc":
            best_val_roc_auc,

        "best_val_accuracy":
            best_val_accuracy,

        "num_epochs_completed":
            NUM_EPOCHS,

        "early_stopping":
            False,

        "test_used":
            False,

        "test_predictions_generated":
            False,

        "seed":
            SEED,

        "model_name":
            "coatnet_0_rw_224"
    },
    FINAL_CHECKPOINT_PATH
)


# ============================================================
# 15. FINAL DEVELOPMENT TARGET SUMMARY
# ============================================================

history_df = pd.DataFrame(
    history
)

target_df = pd.DataFrame(
    target_history
)

target_df.to_csv(
    TARGET_HISTORY_PATH,
    index=False
)

history_df.to_csv(
    TRAINING_LOG_PATH,
    index=False
)

train_gt90_epochs = target_df[
    target_df["train_gt_90"]
]

val_acc_gt90_epochs = target_df[
    target_df["validation_accuracy_gt_90"]
]

val_auc_gt90_epochs = target_df[
    target_df["validation_auc_gt_90"]
]

all_gt90_epochs = target_df[
    target_df["all_targets_gt_90"]
]


# ============================================================
# 16. FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 80)
print("RUN-2 TRAINING COMPLETE")
print("=" * 80)

print(
    f"Epochs completed       : "
    f"{NUM_EPOCHS}"
)

print(
    f"Best epoch             : "
    f"{best_epoch}"
)

print(
    f"Best validation AUC    : "
    f"{best_val_roc_auc:.6f}"
)

print(
    f"Best validation acc.   : "
    f"{best_val_accuracy:.6f}"
)

print(
    f"Total training time    : "
    f"{total_training_time / 3600:.2f} hours"
)

print("\n>90% DEVELOPMENT TARGET SUMMARY")
print("-" * 80)

print(
    f"Train >90% epochs      : "
    f"{len(train_gt90_epochs)}"
)

print(
    f"Val accuracy >90%      : "
    f"{len(val_acc_gt90_epochs)}"
)

print(
    f"Val ROC-AUC >90%       : "
    f"{len(val_auc_gt90_epochs)}"
)

print(
    f"ALL THREE >90%         : "
    f"{len(all_gt90_epochs)}"
)

print("\nTEST SET")
print("-" * 80)

print("Test loaded            : NO")
print("Test evaluated         : NO")
print("Test predictions       : NO")
print("Test threshold tuning  : NO")

print("\nOUTPUTS")
print("-" * 80)

print(
    f"Best checkpoint:\n"
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    f"\nFinal checkpoint:\n"
    f"{FINAL_CHECKPOINT_PATH}"
)

print(
    f"\nTraining log:\n"
    f"{TRAINING_LOG_PATH}"
)

print(
    f"\nTarget tracking:\n"
    f"{TARGET_HISTORY_PATH}"
)

print("\n" + "=" * 80)
print("CELL 4 STATUS: PASS — TRAINING COMPLETED")
print("=" * 80)

NOTEBOOK 10C — CELL 4
CoAtNet-0 RUN-2 FULL TRAINING
Device : cpu

OBJECT VERIFICATION
--------------------------------------------------------------------------------
train_loader        : PASS
val_loader          : PASS
train_df            : PASS
val_df              : PASS

MODEL OBJECT VERIFICATION
--------------------------------------------------------------------------------
model               : PASS
optimizer           : PASS
criterion           : PASS
scheduler           : PASS

TEST ISOLATION
--------------------------------------------------------------------------------
Test dataframe loaded       : NO
Test DataLoader created     : NO
Test predictions generated  : NO
Test accuracy calculated    : NO
Test used for checkpointing : NO
Test used for thresholding  : NO

STARTING RUN-2 TRAINING
Total epochs             : 40
Gradient accumulation    : 4
Effective batch size     : 16
Backbone LR              : 1.00000000e-05
Classifier LR            : 1.00000000e-04
Early stopping  

In [1]:
# =============================================================================
# NOTEBOOK 10G
# CELL 1 — RUN-2 CoAtNet FINAL TEST EVALUATION SAFETY GATE
# =============================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

import timm

print("=" * 80)
print("NOTEBOOK 10G — CELL 1")
print("RUN-2 CoAtNet FINAL TEST EVALUATION")
print("=" * 80)

# =============================================================================
# 1. DEVICE
# =============================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDEVICE")
print("-" * 80)
print("Device:", DEVICE)

# =============================================================================
# 2. AUTHORITATIVE ROOT
# =============================================================================

FINAL_ROOT = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation"
    r"\QC\Clean_PSPNet_CXR_Cohort"
    r"\FINAL_FROZEN_CLEAN_COHORT"
)

SPLIT_ROOT = (
    FINAL_ROOT /
    "MASTER_CONDITION_LEVEL_SPLIT"
)

INPUT_ROOT = (
    FINAL_ROOT /
    "CoAtNet_224_Final_Input"
)

NPY_ROOT = (
    INPUT_ROOT /
    "NPY"
)

MANIFEST_FILE = (
    INPUT_ROOT /
    "QC" /
    "Notebook08_Cell26B_FINAL_1976_CoAtNet_Input_Manifest.csv"
)

TEST_FILE = (
    SPLIT_ROOT /
    "Notebook08_Cell25_TEST.csv"
)

FULL_SPLIT_FILE = (
    SPLIT_ROOT /
    "Notebook08_Cell25_FINAL_1976_Condition_Level_Split.csv"
)

# =============================================================================
# 3. RUN-2 CHECKPOINT
# =============================================================================

RUN2_ROOT = (
    FINAL_ROOT /
    "CoAtNet_Training_Run2_10C"
)

BEST_CHECKPOINT = (
    RUN2_ROOT /
    "checkpoints" /
    "Notebook10C_Run2_Best_Validation_ROC_AUC.pth"
)

TRAINING_LOG = (
    RUN2_ROOT /
    "logs" /
    "Notebook10C_Run2_Training_Log.csv"
)

# =============================================================================
# 4. OUTPUT DIRECTORY
# =============================================================================

OUTPUT_ROOT = (
    FINAL_ROOT /
    "CoAtNet_Test_Evaluation_Run2_10G"
)

CELL1_ROOT = (
    OUTPUT_ROOT /
    "Cell1_Safety_Gate"
)

CELL1_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# =============================================================================
# 5. PATH CHECK
# =============================================================================

print("\nPATH CHECK")
print("-" * 80)

required_paths = {

    "Full frozen split":
        FULL_SPLIT_FILE,

    "Test split":
        TEST_FILE,

    "CoAtNet manifest":
        MANIFEST_FILE,

    "CoAtNet NPY root":
        NPY_ROOT,

    "Run-2 training log":
        TRAINING_LOG,

    "Run-2 best checkpoint":
        BEST_CHECKPOINT,

}

for name, path in required_paths.items():

    status = path.exists()

    print(
        f"{name:<30}: "
        f"{'PASS' if status else 'FAIL'}"
    )

    if not status:

        raise FileNotFoundError(
            f"Missing required artifact:\n{path}"
        )

# =============================================================================
# 6. LOAD AUTHORITATIVE DATA
# =============================================================================

full_df = pd.read_csv(
    FULL_SPLIT_FILE
)

test_df = pd.read_csv(
    TEST_FILE
)

manifest_df = pd.read_csv(
    MANIFEST_FILE
)

for df in [
    full_df,
    test_df,
    manifest_df
]:

    df["condition_id"] = (
        df["condition_id"]
        .astype(str)
        .str.strip()
    )

# =============================================================================
# 7. COHORT GATE
# =============================================================================

print("\nCOHORT GATE")
print("-" * 80)

if len(full_df) != 1976:
    raise RuntimeError(
        "Frozen cohort is not 1976."
    )

if len(test_df) != 396:
    raise RuntimeError(
        "Frozen test split is not 396."
    )

print(
    "Frozen cohort:",
    len(full_df)
)

print(
    "Frozen test:",
    len(test_df)
)

# =============================================================================
# 8. TEST TARGET GATE
# =============================================================================

test_ds = int(
    (
        test_df["target_binary"] == 0
    ).sum()
)

test_dr = int(
    (
        test_df["target_binary"] == 1
    ).sum()
)

print("\nTEST TARGET")
print("-" * 80)

print(
    "DS-TB:",
    test_ds
)

print(
    "DR-TB:",
    test_dr
)

if (
    test_ds,
    test_dr
) != (
    141,
    255
):

    raise RuntimeError(
        "Test target distribution does not match "
        "the frozen split."
    )

print(
    "Target distribution: PASS"
)

# =============================================================================
# 9. TEST CONDITION UNIQUENESS
# =============================================================================

test_ids = set(
    test_df["condition_id"]
)

if len(test_ids) != 396:

    raise RuntimeError(
        "Duplicate condition IDs in test split."
    )

print(
    "Unique test conditions: 396"
)

# =============================================================================
# 10. TEST INPUT GATE
# =============================================================================

missing_inputs = []

for condition_id in test_ids:

    path = (
        NPY_ROOT /
        f"{condition_id}.npy"
    )

    if not path.exists():

        missing_inputs.append(
            condition_id
        )

if missing_inputs:

    raise RuntimeError(
        f"Missing {len(missing_inputs)} "
        f"test NPY files."
    )

print(
    "Physical test NPY files: 396"
)

print(
    "Test input gate: PASS"
)

# =============================================================================
# 11. VERIFY RUN-2 TRAINING LOG
# =============================================================================

print("\nRUN-2 TRAINING RECORD")
print("-" * 80)

training_log = pd.read_csv(
    TRAINING_LOG
)

print(
    "Training epochs:",
    len(training_log)
)

if len(training_log) != 40:

    raise RuntimeError(
        "Run-2 training log does not contain 40 epochs."
    )

best_row = training_log.loc[
    training_log[
        "val_roc_auc"
    ].idxmax()
]

best_epoch_from_log = int(
    best_row["epoch"]
)

best_auc_from_log = float(
    best_row["val_roc_auc"]
)

best_acc_from_log = float(
    best_row["val_accuracy"]
)

print(
    "Best epoch:",
    best_epoch_from_log
)

print(
    "Best validation ROC-AUC:",
    best_auc_from_log
)

print(
    "Validation accuracy at best AUC:",
    best_acc_from_log
)

if best_epoch_from_log != 7:

    raise RuntimeError(
        "Unexpected Run-2 best validation epoch."
    )

if not np.isclose(
    best_auc_from_log,
    0.722930,
    atol=1e-5
):

    raise RuntimeError(
        "Run-2 best validation AUC does not match "
        "the recorded result."
    )

print(
    "Run-2 validation checkpoint record: PASS"
)

# =============================================================================
# 12. CHECKPOINT INSPECTION
# =============================================================================

print("\nCHECKPOINT INSPECTION")
print("-" * 80)

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=DEVICE
)

checkpoint_epoch = int(
    checkpoint["epoch"]
)

checkpoint_stage = checkpoint.get(
    "stage",
    "UNKNOWN"
)

checkpoint_auc = float(
    checkpoint[
        "best_val_roc_auc"
    ]
)

print(
    "Checkpoint epoch:",
    checkpoint_epoch
)

print(
    "Checkpoint stage:",
    checkpoint_stage
)

print(
    "Stored validation ROC-AUC:",
    checkpoint_auc
)

if checkpoint_epoch != 7:

    raise RuntimeError(
        "Checkpoint is not the validation-selected "
        "epoch-7 checkpoint."
    )

if not np.isclose(
    checkpoint_auc,
    0.722930,
    atol=1e-5
):

    raise RuntimeError(
        "Checkpoint validation AUC mismatch."
    )

if checkpoint.get(
    "test_used",
    True
):

    raise RuntimeError(
        "Checkpoint indicates test data was used."
    )

print(
    "Checkpoint test_used: False"
)

print(
    "Checkpoint safety: PASS"
)

# =============================================================================
# 13. CREATE CoAtNet
# =============================================================================

print("\nMODEL")
print("-" * 80)

MODEL_NAME = "coatnet_0_rw_224"

model = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=2
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True
)

model = model.to(
    DEVICE
)

model.eval()

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "Model:",
    MODEL_NAME
)

print(
    "Class:",
    model.__class__.__name__
)

print(
    "Parameters:",
    f"{total_params:,}"
)

if total_params != 26_668_100:

    raise RuntimeError(
        "Unexpected CoAtNet parameter count."
    )

print(
    "Model loading: PASS"
)

# =============================================================================
# 14. TEST DATASET
# =============================================================================

class Run2TestDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        npy_root
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.npy_root = Path(
            npy_root
        )

    def __len__(
        self
    ):

        return len(
            self.df
        )

    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[
            index
        ]

        condition_id = str(
            row["condition_id"]
        )

        label = int(
            row["target_binary"]
        )

        npy_path = (
            self.npy_root /
            f"{condition_id}.npy"
        )

        image = np.load(
            npy_path
        )

        if image.shape != (
            224,
            224,
            3
        ):

            raise RuntimeError(
                f"Invalid shape for "
                f"{condition_id}: "
                f"{image.shape}"
            )

        image = image.astype(
            np.float32,
            copy=False
        )

        if not np.isfinite(
            image
        ).all():

            raise RuntimeError(
                f"NaN/Inf detected: "
                f"{condition_id}"
            )

        if (
            image.min() < -1.00001
            or
            image.max() > 1.00001
        ):

            raise RuntimeError(
                f"Invalid stored range for "
                f"{condition_id}: "
                f"{image.min()} to "
                f"{image.max()}"
            )

        # -------------------------------------------------------------
        # Stored representation:
        # [-1, 1]
        #
        # Convert to:
        # [0, 1]
        # -------------------------------------------------------------

        image = (
            image + 1.0
        ) / 2.0

        image = np.clip(
            image,
            0.0,
            1.0
        )

        # HWC -> CHW
        image = torch.from_numpy(
            image
        ).permute(
            2,
            0,
            1
        ).contiguous()

        # -------------------------------------------------------------
        # EXACT ImageNet normalization used by Run-2
        # -------------------------------------------------------------

        mean = torch.tensor(
            [
                0.485,
                0.456,
                0.406
            ],
            dtype=torch.float32
        ).view(
            3,
            1,
            1
        )

        std = torch.tensor(
            [
                0.229,
                0.224,
                0.225
            ],
            dtype=torch.float32
        ).view(
            3,
            1,
            1
        )

        image = (
            image - mean
        ) / std

        return (
            image,
            torch.tensor(
                label,
                dtype=torch.long
            ),
            condition_id
        )


# =============================================================================
# 15. TEST DATALOADER
# =============================================================================

test_dataset = Run2TestDataset(
    test_df,
    NPY_ROOT
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    drop_last=False
)

print("\nTEST DATALOADER")
print("-" * 80)

print(
    "Test samples:",
    len(test_dataset)
)

print(
    "Test batches:",
    len(test_loader)
)

print(
    "Shuffle:",
    False
)

# =============================================================================
# 16. SAMPLE QC
# =============================================================================

sample_image, sample_label, sample_id = (
    test_dataset[0]
)

print("\nTEST SAMPLE QC")
print("-" * 80)

print(
    "Condition:",
    sample_id
)

print(
    "Shape:",
    tuple(
        sample_image.shape
    )
)

print(
    "Dtype:",
    sample_image.dtype
)

print(
    "Label:",
    int(sample_label)
)

print(
    "Finite:",
    bool(
        torch.isfinite(
            sample_image
        ).all()
    )
)

if sample_image.shape != (
    3,
    224,
    224
):

    raise RuntimeError(
        "Test sample shape check failed."
    )

if not torch.isfinite(
    sample_image
).all():

    raise RuntimeError(
        "Test sample finite-value check failed."
    )

# =============================================================================
# 17. HARD TEST ISOLATION STATEMENT
# =============================================================================

print("\nTEST ISOLATION")
print("-" * 80)

print(
    "Training with test set       : NO"
)

print(
    "Checkpoint selection on test : NO"
)

print(
    "Threshold selection on test  : NO"
)

print(
    "Test augmentation             : NO"
)

print(
    "Test evaluation before now    : NO"
)

print(
    "Current purpose               : FINAL EVALUATION"
)

# =============================================================================
# 18. SAVE CONFIGURATION
# =============================================================================

config = {

    "notebook":
        "10G_CXR_CoAtNet_Run2_Test_Evaluation",

    "model":
        MODEL_NAME,

    "representation":
        "CoAtNet_Run2_final_1976_CXR_input",

    "checkpoint_epoch":
        checkpoint_epoch,

    "checkpoint_stage":
        checkpoint_stage,

    "validation_auc":
        checkpoint_auc,

    "test_samples":
        396,

    "test_DS":
        141,

    "test_DR":
        255,

    "batch_size":
        4,

    "threshold":
        0.50,

    "augmentation":
        False,

    "test_used_for_training":
        False,

    "test_used_for_model_selection":
        False,

    "test_used_for_threshold_selection":
        False,

    "status":
        "PASS"

}

CONFIG_FILE = (
    CELL1_ROOT /
    "Notebook10G_Cell1_Run2_Test_Evaluation_Config.json"
)

with open(
    CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )

# =============================================================================
# 19. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 1 STATUS: PASS")
print("=" * 80)

print(
    "Frozen test conditions:",
    396
)

print(
    "Test DS:",
    141
)

print(
    "Test DR:",
    255
)

print(
    "Run-2 best checkpoint epoch:",
    checkpoint_epoch
)

print(
    "Run-2 best validation ROC-AUC:",
    f"{checkpoint_auc:.6f}"
)

print(
    "Test previously evaluated:",
    "NO"
)

print(
    "Test tuning:",
    "NO"
)

print(
    "Ready for final Run-2 test prediction:",
    "YES"
)

print(
    "\nConfiguration:"
)

print(
    CONFIG_FILE
)

NOTEBOOK 10G — CELL 1
RUN-2 CoAtNet FINAL TEST EVALUATION

DEVICE
--------------------------------------------------------------------------------
Device: cpu

PATH CHECK
--------------------------------------------------------------------------------
Full frozen split             : PASS
Test split                    : PASS
CoAtNet manifest              : PASS
CoAtNet NPY root              : PASS
Run-2 training log            : PASS
Run-2 best checkpoint         : PASS

COHORT GATE
--------------------------------------------------------------------------------
Frozen cohort: 1976
Frozen test: 396

TEST TARGET
--------------------------------------------------------------------------------
DS-TB: 141
DR-TB: 255
Target distribution: PASS
Unique test conditions: 396
Physical test NPY files: 396
Test input gate: PASS

RUN-2 TRAINING RECORD
--------------------------------------------------------------------------------
Training epochs: 40
Best epoch: 7
Best validation ROC-AUC: 0.722929580

In [2]:
# =============================================================================
# NOTEBOOK 10G — CELL 2
# RUN-2 CoAtNet — FINAL 396 TEST PREDICTIONS
# =============================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print("=" * 80)
print("NOTEBOOK 10G — CELL 2")
print("RUN-2 CoAtNet — FINAL 396 TEST PREDICTIONS")
print("=" * 80)

# =============================================================================
# 1. OBJECT SAFETY
# =============================================================================

print("\nOBJECT SAFETY")
print("-" * 80)

for name, obj in {
    "model": model,
    "test_loader": test_loader,
    "test_dataset": test_dataset,
    "DEVICE": DEVICE,
}.items():

    if obj is None:
        raise RuntimeError(
            f"Required object missing: {name}"
        )

    print(
        f"{name:<20}: PASS"
    )

model.eval()

# =============================================================================
# 2. PREDICTION CONTAINERS
# =============================================================================

condition_ids = []
true_labels = []
dr_probabilities = []
predicted_labels = []

# =============================================================================
# 3. FINAL TEST INFERENCE
# =============================================================================

print("\nGENERATING 396 TEST PREDICTIONS")
print("-" * 80)

with torch.no_grad():

    for batch_number, batch in enumerate(
        test_loader,
        start=1
    ):

        images, labels, ids = batch

        images = images.to(
            DEVICE
        )

        labels = labels.to(
            DEVICE
        )

        outputs = model(
            images
        )

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        dr_probability = (
            probabilities[:, 1]
            .cpu()
            .numpy()
        )

        predictions = (
            dr_probability >= 0.50
        ).astype(
            np.int64
        )

        labels_np = (
            labels
            .cpu()
            .numpy()
            .astype(
                np.int64
            )
        )

        condition_ids.extend(
            [str(x) for x in ids]
        )

        true_labels.extend(
            labels_np.tolist()
        )

        dr_probabilities.extend(
            dr_probability.tolist()
        )

        predicted_labels.extend(
            predictions.tolist()
        )

        if (
            batch_number == 1
            or
            batch_number % 20 == 0
            or
            batch_number == len(test_loader)
        ):

            print(
                f"Processed "
                f"{len(condition_ids):>3}/396 "
                f"| batch "
                f"{batch_number:>3}/{len(test_loader)}"
            )

# =============================================================================
# 4. COUNT GATE
# =============================================================================

print("\nPREDICTION COUNT GATE")
print("-" * 80)

if len(condition_ids) != 396:

    raise RuntimeError(
        f"Expected 396 predictions, "
        f"got {len(condition_ids)}."
    )

print(
    "Predictions generated: 396"
)

# =============================================================================
# 5. CONDITION UNIQUENESS
# =============================================================================

if len(
    set(condition_ids)
) != 396:

    raise RuntimeError(
        "Duplicate or missing condition IDs."
    )

print(
    "Unique conditions: 396"
)

# =============================================================================
# 6. ARRAY VALIDATION
# =============================================================================

y_true = np.asarray(
    true_labels,
    dtype=np.int64
)

y_prob = np.asarray(
    dr_probabilities,
    dtype=np.float64
)

y_pred = np.asarray(
    predicted_labels,
    dtype=np.int64
)

if not np.isfinite(
    y_prob
).all():

    raise RuntimeError(
        "Non-finite probabilities detected."
    )

if not (
    (y_prob >= 0)
    &
    (y_prob <= 1)
).all():

    raise RuntimeError(
        "Probability outside [0,1]."
    )

if not np.isin(
    y_true,
    [0, 1]
).all():

    raise RuntimeError(
        "Invalid true labels."
    )

if not np.isin(
    y_pred,
    [0, 1]
).all():

    raise RuntimeError(
        "Invalid predictions."
    )

print("\nPREDICTION VALIDITY")
print("-" * 80)

print(
    "Finite probabilities: PASS"
)

print(
    "Probability range: [0,1] PASS"
)

print(
    "Binary predictions: PASS"
)

# =============================================================================
# 7. VERIFY AGAINST FROZEN TEST SPLIT
# =============================================================================

frozen_lookup = dict(
    zip(
        test_df[
            "condition_id"
        ].astype(str),
        test_df[
            "target_binary"
        ].astype(int)
    )
)

for condition_id, label in zip(
    condition_ids,
    y_true
):

    expected = frozen_lookup[
        condition_id
    ]

    if int(label) != int(expected):

        raise RuntimeError(
            f"Frozen target mismatch: "
            f"{condition_id}"
        )

print(
    "All 396 labels match frozen test split: PASS"
)

# =============================================================================
# 8. CREATE PREDICTION TABLE
# =============================================================================

prediction_df = pd.DataFrame({

    "condition_id":
        condition_ids,

    "target_binary":
        y_true,

    "true_class":
        np.where(
            y_true == 1,
            "DR-TB",
            "DS-TB"
        ),

    "probability_DS":
        1.0 - y_prob,

    "probability_DR":
        y_prob,

    "predicted_binary":
        y_pred,

    "predicted_class":
        np.where(
            y_pred == 1,
            "DR-TB",
            "DS-TB"
        ),

    "decision_threshold":
        0.50,

})

# =============================================================================
# 9. DISTRIBUTION
# =============================================================================

actual_ds = int(
    (y_true == 0).sum()
)

actual_dr = int(
    (y_true == 1).sum()
)

predicted_ds = int(
    (y_pred == 0).sum()
)

predicted_dr = int(
    (y_pred == 1).sum()
)

print("\nPREDICTION DISTRIBUTION")
print("-" * 80)

print(
    "Actual DS-TB:",
    actual_ds
)

print(
    "Actual DR-TB:",
    actual_dr
)

print(
    "Predicted DS-TB:",
    predicted_ds
)

print(
    "Predicted DR-TB:",
    predicted_dr
)

if (
    actual_ds,
    actual_dr
) != (
    141,
    255
):

    raise RuntimeError(
        "Frozen actual test distribution changed."
    )

# =============================================================================
# 10. OUTPUT PATHS
# =============================================================================

OUTPUT_ROOT = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation"
    r"\QC\Clean_PSPNet_CXR_Cohort"
    r"\FINAL_FROZEN_CLEAN_COHORT"
    r"\CoAtNet_Test_Evaluation_Run2_10G"
)

PREDICTION_ROOT = (
    OUTPUT_ROOT /
    "Cell2_Final_Test_Predictions"
)

PREDICTION_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

PREDICTION_FILE = (
    PREDICTION_ROOT /
    "Notebook10G_Cell2_Run2_FINAL_396_Test_Predictions.csv"
)

SUMMARY_FILE = (
    PREDICTION_ROOT /
    "Notebook10G_Cell2_Run2_Test_Prediction_Summary.json"
)

# =============================================================================
# 11. SAVE PREDICTIONS
# =============================================================================

prediction_df.to_csv(
    PREDICTION_FILE,
    index=False
)

summary = {

    "notebook":
        "10G_CXR_CoAtNet_Run2_Test_Evaluation",

    "model":
        "coatnet_0_rw_224",

    "checkpoint_epoch":
        7,

    "validation_roc_auc":
        0.7229295806109344,

    "test_samples":
        396,

    "actual_DS":
        actual_ds,

    "actual_DR":
        actual_dr,

    "predicted_DS":
        predicted_ds,

    "predicted_DR":
        predicted_dr,

    "threshold":
        0.50,

    "test_training":
        False,

    "test_checkpoint_selection":
        False,

    "test_threshold_tuning":
        False,

    "status":
        "PASS"

}

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

# =============================================================================
# 12. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 2 STATUS: PASS")
print("=" * 80)

print(
    "Test conditions:",
    396
)

print(
    "Actual DS-TB:",
    actual_ds
)

print(
    "Actual DR-TB:",
    actual_dr
)

print(
    "Predicted DS-TB:",
    predicted_ds
)

print(
    "Predicted DR-TB:",
    predicted_dr
)

print(
    "Threshold:",
    0.50
)

print(
    "Test tuning: NO"
)

print(
    "Model modification: NO"
)

print(
    "\nPrediction file:"
)

print(
    PREDICTION_FILE
)

print(
    "\nSummary file:"
)

print(
    SUMMARY_FILE
)

print(
    "\nReady for Cell 3 — independent Run-2 test metrics."
)

NOTEBOOK 10G — CELL 2
RUN-2 CoAtNet — FINAL 396 TEST PREDICTIONS

OBJECT SAFETY
--------------------------------------------------------------------------------
model               : PASS
test_loader         : PASS
test_dataset        : PASS
DEVICE              : PASS

GENERATING 396 TEST PREDICTIONS
--------------------------------------------------------------------------------
Processed   4/396 | batch   1/99
Processed  80/396 | batch  20/99
Processed 160/396 | batch  40/99
Processed 240/396 | batch  60/99
Processed 320/396 | batch  80/99
Processed 396/396 | batch  99/99

PREDICTION COUNT GATE
--------------------------------------------------------------------------------
Predictions generated: 396
Unique conditions: 396

PREDICTION VALIDITY
--------------------------------------------------------------------------------
Finite probabilities: PASS
Probability range: [0,1] PASS
Binary predictions: PASS
All 396 labels match frozen test split: PASS

PREDICTION DISTRIBUTION
-----------

In [3]:
# =============================================================================
# NOTEBOOK 10G — CELL 3
# RUN-2 CoAtNet — FINAL INDEPENDENT TEST METRICS
# =============================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

print("=" * 80)
print("NOTEBOOK 10G — CELL 3")
print("RUN-2 CoAtNet — FINAL INDEPENDENT TEST METRICS")
print("=" * 80)

# =============================================================================
# 1. LOAD PREDICTIONS
# =============================================================================

PREDICTION_FILE = Path(
    r"C:\TBP\Metadata\Step_3B_8_CXR_CoAtNet_Preparation"
    r"\QC\Clean_PSPNet_CXR_Cohort"
    r"\FINAL_FROZEN_CLEAN_COHORT"
    r"\CoAtNet_Test_Evaluation_Run2_10G"
    r"\Cell2_Final_Test_Predictions"
    r"\Notebook10G_Cell2_Run2_FINAL_396_Test_Predictions.csv"
)

OUTPUT_ROOT = (
    PREDICTION_FILE.parent.parent /
    "Cell3_Final_Test_Metrics"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

if not PREDICTION_FILE.exists():

    raise FileNotFoundError(
        f"Prediction file not found:\n{PREDICTION_FILE}"
    )

prediction_df = pd.read_csv(
    PREDICTION_FILE
)

print("\nPREDICTION FILE")
print("-" * 80)

print(
    "Rows:",
    len(prediction_df)
)

if len(prediction_df) != 396:

    raise RuntimeError(
        "Expected exactly 396 test predictions."
    )

# =============================================================================
# 2. REQUIRED COLUMNS
# =============================================================================

required_columns = [

    "condition_id",
    "target_binary",
    "true_class",
    "probability_DS",
    "probability_DR",
    "predicted_binary",
    "predicted_class",
    "decision_threshold",

]

missing = [
    c
    for c in required_columns
    if c not in prediction_df.columns
]

if missing:

    raise RuntimeError(
        f"Missing columns: {missing}"
    )

print(
    "Required columns: PASS"
)

# =============================================================================
# 3. CONDITION UNIQUENESS
# =============================================================================

if prediction_df[
    "condition_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate condition IDs detected."
    )

print(
    "Unique conditions: 396"
)

# =============================================================================
# 4. ARRAYS
# =============================================================================

y_true = (
    prediction_df[
        "target_binary"
    ]
    .astype(int)
    .to_numpy()
)

y_prob = (
    prediction_df[
        "probability_DR"
    ]
    .astype(float)
    .to_numpy()
)

y_pred = (
    prediction_df[
        "predicted_binary"
    ]
    .astype(int)
    .to_numpy()
)

thresholds = (
    prediction_df[
        "decision_threshold"
    ]
    .astype(float)
    .unique()
)

if len(thresholds) != 1:

    raise RuntimeError(
        "Multiple decision thresholds found."
    )

threshold = float(
    thresholds[0]
)

# =============================================================================
# 5. VALIDITY
# =============================================================================

print("\nDATA VALIDITY")
print("-" * 80)

if not np.isfinite(
    y_prob
).all():

    raise RuntimeError(
        "Non-finite probabilities."
    )

if not (
    (y_prob >= 0)
    &
    (y_prob <= 1)
).all():

    raise RuntimeError(
        "Probabilities outside [0,1]."
    )

if not np.isin(
    y_true,
    [0, 1]
).all():

    raise RuntimeError(
        "Invalid true labels."
    )

if not np.isin(
    y_pred,
    [0, 1]
).all():

    raise RuntimeError(
        "Invalid predictions."
    )

print(
    "True labels: PASS"
)

print(
    "Probabilities: PASS"
)

print(
    "Predictions: PASS"
)

print(
    "Threshold:",
    threshold
)

# =============================================================================
# 6. CONFUSION MATRIX
# =============================================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = (
    cm.ravel()
)

print("\nCONFUSION MATRIX")
print("-" * 80)

print(
    "                 Predicted"
)

print(
    "                 DS    DR"
)

print(
    f"Actual DS       {tn:3d}   {fp:3d}"
)

print(
    f"Actual DR       {fn:3d}   {tp:3d}"
)

# =============================================================================
# 7. METRICS
# =============================================================================

accuracy = accuracy_score(
    y_true,
    y_pred
)

balanced_accuracy = (
    balanced_accuracy_score(
        y_true,
        y_pred
    )
)

precision = precision_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

sensitivity = recall_score(
    y_true,
    y_pred,
    pos_label=1,
    zero_division=0
)

specificity = recall_score(
    y_true,
    y_pred,
    pos_label=0,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_true,
    y_prob
)

pr_auc = average_precision_score(
    y_true,
    y_prob
)

# =============================================================================
# 8. PRINT RESULTS
# =============================================================================

print("\nFINAL INDEPENDENT TEST METRICS")
print("-" * 80)

print(
    f"Accuracy             : "
    f"{accuracy:.6f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Balanced Accuracy    : "
    f"{balanced_accuracy:.6f} "
    f"({balanced_accuracy * 100:.2f}%)"
)

print(
    f"Precision            : "
    f"{precision:.6f}"
)

print(
    f"Sensitivity / Recall : "
    f"{sensitivity:.6f}"
)

print(
    f"Specificity          : "
    f"{specificity:.6f}"
)

print(
    f"F1 Score             : "
    f"{f1:.6f}"
)

print(
    f"ROC-AUC              : "
    f"{roc_auc:.6f}"
)

print(
    f"PR-AUC               : "
    f"{pr_auc:.6f}"
)

# =============================================================================
# 9. >90% CHECK
# =============================================================================

print("\nTARGET CHECK")
print("-" * 80)

print(
    "Accuracy >90%:",
    "YES" if accuracy > 0.90 else "NO"
)

print(
    "ROC-AUC >90%:",
    "YES" if roc_auc > 0.90 else "NO"
)

# =============================================================================
# 10. SAVE METRICS
# =============================================================================

metrics_df = pd.DataFrame({

    "metric": [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "sensitivity",
        "specificity",
        "f1",
        "roc_auc",
        "pr_auc",
    ],

    "value": [
        accuracy,
        balanced_accuracy,
        precision,
        sensitivity,
        specificity,
        f1,
        roc_auc,
        pr_auc,
    ],

    "percentage": [
        accuracy * 100,
        balanced_accuracy * 100,
        precision * 100,
        sensitivity * 100,
        specificity * 100,
        f1 * 100,
        roc_auc * 100,
        pr_auc * 100,
    ],

})

METRICS_FILE = (
    OUTPUT_ROOT /
    "Notebook10G_Cell3_Run2_FINAL_Test_Metrics.csv"
)

metrics_df.to_csv(
    METRICS_FILE,
    index=False
)

# =============================================================================
# 11. SAVE CONFUSION MATRIX
# =============================================================================

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual_DS",
        "Actual_DR"
    ],
    columns=[
        "Predicted_DS",
        "Predicted_DR"
    ]
)

CM_FILE = (
    OUTPUT_ROOT /
    "Notebook10G_Cell3_Run2_FINAL_Confusion_Matrix.csv"
)

cm_df.to_csv(
    CM_FILE
)

# =============================================================================
# 12. SAVE JSON
# =============================================================================

results = {

    "notebook":
        "10G_CXR_CoAtNet_Run2_Test_Evaluation",

    "model":
        "coatnet_0_rw_224",

    "checkpoint_epoch":
        7,

    "validation_roc_auc":
        0.7229295806109344,

    "test_samples":
        396,

    "actual_DS":
        int((y_true == 0).sum()),

    "actual_DR":
        int((y_true == 1).sum()),

    "predicted_DS":
        int((y_pred == 0).sum()),

    "predicted_DR":
        int((y_pred == 1).sum()),

    "threshold":
        threshold,

    "TN":
        int(tn),

    "FP":
        int(fp),

    "FN":
        int(fn),

    "TP":
        int(tp),

    "accuracy":
        float(accuracy),

    "balanced_accuracy":
        float(balanced_accuracy),

    "precision":
        float(precision),

    "sensitivity":
        float(sensitivity),

    "specificity":
        float(specificity),

    "f1":
        float(f1),

    "roc_auc":
        float(roc_auc),

    "pr_auc":
        float(pr_auc),

    "accuracy_gt_90":
        bool(accuracy > 0.90),

    "roc_auc_gt_90":
        bool(roc_auc > 0.90),

    "test_used_for_training":
        False,

    "test_used_for_model_selection":
        False,

    "test_used_for_threshold_selection":
        False,

    "status":
        "PASS"

}

JSON_FILE = (
    OUTPUT_ROOT /
    "Notebook10G_Cell3_Run2_FINAL_Test_Metrics.json"
)

with open(
    JSON_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )

# =============================================================================
# 13. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 3 STATUS: PASS")
print("=" * 80)

print(
    f"FINAL RUN-2 TEST ACCURACY : "
    f"{accuracy * 100:.2f}%"
)

print(
    f"FINAL RUN-2 TEST ROC-AUC : "
    f"{roc_auc:.4f}"
)

print(
    f"FINAL RUN-2 TEST PR-AUC  : "
    f"{pr_auc:.4f}"
)

print(
    f"CONFUSION MATRIX: "
    f"TN={tn}, FP={fp}, FN={fn}, TP={tp}"
)

print(
    "\nMetrics:",
    METRICS_FILE
)

print(
    "\nConfusion matrix:",
    CM_FILE
)

print(
    "\nJSON:",
    JSON_FILE
)

print(
    "\nRun-2 independent test evaluation completed."
)

NOTEBOOK 10G — CELL 3
RUN-2 CoAtNet — FINAL INDEPENDENT TEST METRICS

PREDICTION FILE
--------------------------------------------------------------------------------
Rows: 396
Required columns: PASS
Unique conditions: 396

DATA VALIDITY
--------------------------------------------------------------------------------
True labels: PASS
Probabilities: PASS
Predictions: PASS
Threshold: 0.5

CONFUSION MATRIX
--------------------------------------------------------------------------------
                 Predicted
                 DS    DR
Actual DS        58    83
Actual DR        39   216

FINAL INDEPENDENT TEST METRICS
--------------------------------------------------------------------------------
Accuracy             : 0.691919 (69.19%)
Balanced Accuracy    : 0.629203 (62.92%)
Precision            : 0.722408
Sensitivity / Recall : 0.847059
Specificity          : 0.411348
F1 Score             : 0.779783
ROC-AUC              : 0.715589
PR-AUC               : 0.797732

TARGET CHECK
-----